# VOSSA streaming inference

Converts a source utterance to a target speaker chunk by chunk, the way a live application would. Audio is pushed in small blocks and converted audio comes back as soon as each chunk is ready.

For whole-file (offline) conversion, see `inference.ipynb`. For how the streaming step works internally, see `src/streaming/step.py`.

In [1]:
import sys
sys.path.append('..')
import time
import numpy as np
import torch
import librosa
from IPython import display as disp
from src.pretrained import load_vossa
from src.streaming import VossaStreamer

/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [2]:
# Downloads the checkpoints from the Hugging Face Hub on first use.
# To use files already on disk: load_vossa(local_dir="../checkpoints")
resynthesizer = load_vossa()

/data/mtseng/miniconda3/envs/vossa/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loss functions:  {'mel_fine_loss': 10.0, 'mel_coarse_loss': 2.0, 'adv_Gx_loss': 1.0, 'adv_Gx_feat_loss': 5.0, 'speaker_recon_loss': 30.0, 'speaker_vc_loss': 30.0, 'contrastive_loss': 10.0}


## Streaming settings

- `CHUNK_FRAMES`: latent frames converted per step. One frame is 20 ms (320 samples at 16 kHz).
- `LOOKAHEAD`: future frames the content encoder sees. `None` uses the encoder's trained `right_context` (4 frames). More lookahead can improve quality but adds delay.

Build the streamer once per model. `warmup()` captures the CUDA graph so the first real stream is not slowed down.

In [3]:
CHUNK_FRAMES = 3
LOOKAHEAD = 0

streamer = VossaStreamer(resynthesizer, chunk_frames=CHUNK_FRAMES, lookahead=LOOKAHEAD)
streamer.warmup()

step = streamer.step
chunk_ms = 1000 * step.chunk_samples / streamer.sample_rate
print(f"chunk: {step.chunk_samples} samples = {chunk_ms:.0f} ms, lookahead: {step.lookahead} frames")
print(f"buffering delay (excluding compute): {(1 + step.lag_chunks) * chunk_ms:.0f} ms")

chunk: 960 samples = 60 ms, lookahead: 0 frames
buffering delay (excluding compute): 60 ms


## Source and target audio

In [4]:
src_audio, _ = librosa.load("../samples/14_208_000001_000000.wav", sr=16000)
tgt_audio, _ = librosa.load("../samples/arctic_a0002.wav", sr=16000)
disp.display(disp.Audio(src_audio, rate=16000))
disp.display(disp.Audio(tgt_audio, rate=16000))

## Convert as a stream

Open one session per stream and target speaker. `push()` accepts blocks of any size and returns whatever converted audio is ready (possibly nothing). `flush()` returns the rest, so the total output has the same length as the input.

In [5]:
BLOCK = 512  # simulate a microphone delivering 512-sample blocks

session = streamer.session(tgt_audio)
outs = []
for i in range(0, len(src_audio), BLOCK):
    outs.append(session.push(src_audio[i:i + BLOCK]))
outs.append(session.flush())
stream_audio = np.concatenate(outs)

disp.Audio(stream_audio, rate=16000)

## Compute time per chunk

Times one full step per chunk, CNN encoder through vocoder, including the copy back to the CPU. Real-time requires compute below the chunk duration.

In [6]:
session = streamer.session(tgt_audio)
chunks = [src_audio[i:i + step.chunk_samples] for i in range(0, len(src_audio) - step.chunk_samples + 1, step.chunk_samples)]

times_ms = []
for chunk in chunks:
    t0 = time.perf_counter()
    session.push(chunk)  # returns numpy, so the GPU work has finished when it returns
    times_ms.append(1000 * (time.perf_counter() - t0))
session.close()

times_ms = np.array(times_ms[5:])  # skip the first few steps
print(f"compute per {chunk_ms:.0f} ms chunk: mean {times_ms.mean():.1f} ms, p95 {np.percentile(times_ms, 95):.1f} ms")
print(f"real-time factor: {times_ms.mean() / chunk_ms:.3f}")

compute per 60 ms chunk: mean 4.3 ms, p95 4.3 ms
real-time factor: 0.072


## More options

- **Other input sample rates:** `streamer.session(tgt_audio, sample_rate=44100)` resamples pushed audio on the fly. Output is always 16 kHz.
- **Reuse a speaker:** `embed = streamer.speaker_embedding(tgt_audio)`, then `streamer.session(embed)` skips recomputing it.
- **Several streams at once:** each session holds its own state, so many sessions can share one `streamer`.
- **Restart or stop:** `session.reset()` starts a new stream with the same speaker; `session.close()` frees its state.